# 單變量假設檢定自動化模組

## 工具目的

對多個數值變數**自動選擇並執行正確的假設檢定**，根據資料特性（常態性、方差齊性、組數）決定最適合的統計檢定方法。

## 使用場景

比較不同組別（如 Pclass = 1, 2, 3）在數值特徵（如 Fare, Age）上是否存在顯著差異。

## 核心價值

- 避免手動逐一判斷 **常態性 → 方差齊性 → 選擇檢定**，減少人為錯誤
- 自動化流程確保分析的**一致性與可重複性**
- 整合 **Bonferroni 多重比較校正**，避免因多次檢定導致的偽陽性膨脹

## 檢定選擇決策樹

以下是本模組自動選擇檢定方法的完整決策邏輯：

```
判斷組數
├── 2 組
│   ├── 常態性檢定 (Shapiro-Wilk, α=0.05)
│   │   ├── 兩組皆通過 → 方差齊性檢定 (Levene)
│   │   │   ├── 齊性 → Student's t 檢定
│   │   │   └── 不齊 → Welch's t 檢定
│   │   └── 任一組未通過 → Mann-Whitney U 檢定
├── > 2 組
│   ├── 所有組常態 (Shapiro-Wilk, 逐組檢定)
│   │   ├── 全部通過 → 方差齊性檢定 (Levene)
│   │   │   ├── 齊性 → One-way ANOVA
│   │   │   └── 不齊 → Kruskal-Wallis 檢定 (*)
│   │   └── 任一組未通過 → Kruskal-Wallis 檢定
```

### 各檢定方法說明

| 檢定方法 | 適用條件 | 假設 |
|:---|:---|:---|
| **Student's t** | 2 組、常態、方差齊性 | 兩組平均數相等 |
| **Welch's t** | 2 組、常態、方差不齊 | 兩組平均數相等（不假設等方差）|
| **Mann-Whitney U** | 2 組、非常態 | 兩組分佈相同 |
| **One-way ANOVA** | >2 組、常態、方差齊性 | 所有組平均數相等 |
| **Kruskal-Wallis** | >2 組、非常態或方差不齊 | 所有組分佈相同 |

> (*) **設計決策：** 當常態性通過但 Levene 檢定失敗（方差不齊）時，理論上可使用 Welch's ANOVA（`scipy.stats.alexandergovern` 或 `pingouin.welch_anova`）。本模組選擇 Kruskal-Wallis 作為較保守且不需額外套件的替代方案。

### 效應量 (Effect Size)

| 效應量 | 適用場景 | 解讀 |
|:---|:---|:---|
| **Cohen's d** | 2 組參數/非參數檢定 | 小: 0.2, 中: 0.5, 大: 0.8 |
| **Eta-squared (η²)** | One-way ANOVA | 小: 0.01, 中: 0.06, 大: 0.14 |
| **Epsilon-squared (ε²)** | Kruskal-Wallis | 小: 0.01, 中: 0.06, 大: 0.14 |

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
from itertools import combinations
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, mannwhitneyu, kruskal, f_oneway
from statsmodels.stats.multitest import multipletests
import seaborn as sns
import matplotlib.pyplot as plt

# Import shared utility from utils module
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'utils'))
from stat_helpers import create_codebook

# ---- 中文字型設定（解決 CJK glyph missing 警告）----
sys.path.insert(0, os.path.join(
    os.path.dirname(os.path.abspath('__file__')),
    '..', '..', '..', '..', 'Special-Edition_python_DA', 'Python_DA_Course', 'common',
))
from font_setup import setup_chinese_font
setup_chinese_font()

## UnivariateAnalysisModule 類別定義

此類別封裝了完整的單變量假設檢定流程：

1. **`normality_test(data)`** — 對單一組別執行 Shapiro-Wilk 常態性檢定
2. **`variance_homogeneity_test(group_data)`** — 對多組資料執行 Levene 方差齊性檢定
3. **`cohen_d(x, y)`** — 計算兩組之間的 Cohen's d 效應量
4. **`analyze_variable(df, group_col, var)`** — 根據決策樹自動選擇並執行檢定
5. **`run_analysis(...)`** — 批次分析多個變數，並套用 Bonferroni 校正

In [ ]:
class UnivariateAnalysisModule:
    """單變量假設檢定自動化模組

    根據組數、常態性、方差齊性自動選擇適當的統計檢定，
    並計算效應量與 Bonferroni 多重比較校正。
    """

    def __init__(self, alpha=0.05):
        self.alpha = alpha
        self.results = []

    def normality_test(self, data):
        """Shapiro-Wilk normality test for a single group."""
        if len(data) < 3:
            return False, np.nan
        stat, p_value = stats.shapiro(data)
        return p_value > self.alpha, p_value

    def variance_homogeneity_test(self, group_data):
        """Levene's test for equality of variances."""
        if len(group_data) < 2:
            return np.nan
        return stats.levene(*group_data)[1]

    def cohen_d(self, x, y):
        """Cohen's d effect size (pooled std)."""
        pooled_std = np.sqrt((x.var() + y.var()) / 2)
        if pooled_std == 0:
            return 0.0
        return (x.mean() - y.mean()) / pooled_std

    def analyze_variable(self, df, group_col, var):
        """Analyze a single variable: auto-select test based on decision tree."""
        groups = sorted(df[group_col].dropna().unique())
        group_data = [df[df[group_col] == g][var].dropna() for g in groups]

        # Filter out groups with insufficient data
        valid = [(g, d) for g, d in zip(groups, group_data) if len(d) >= 3]
        if len(valid) < 2:
            return  # Skip variables without enough groups

        groups, group_data = zip(*valid)
        groups = list(groups)
        group_data = list(group_data)
        n_groups = len(groups)

        # ---- Shapiro-Wilk normality test for ALL groups ----
        shapiro_results = {}
        all_normal = True
        for g, data in zip(groups, group_data):
            _, sp = stats.shapiro(data)
            shapiro_results[g] = round(sp, 6)
            if sp <= self.alpha:
                all_normal = False

        # ---- 2-group case ----
        if n_groups == 2:
            data_g1, data_g2 = group_data

            if all_normal:
                levene_p = self.variance_homogeneity_test(group_data)
                if levene_p > self.alpha:
                    stat_val, p_value = stats.ttest_ind(data_g1, data_g2, equal_var=True)
                    test_used = "Student's t 檢定"
                else:
                    stat_val, p_value = stats.ttest_ind(data_g1, data_g2, equal_var=False)
                    test_used = "Welch's t 檢定"
                effect_size = self.cohen_d(data_g1, data_g2)
                effect_size_type = "Cohen's d"
            else:
                levene_p = 'N/A (常態性未通過)'
                stat_val, p_value = stats.mannwhitneyu(data_g1, data_g2, alternative='two-sided')
                test_used = "Mann-Whitney U 檢定"
                # Rank-biserial correlation as effect size
                n1, n2 = len(data_g1), len(data_g2)
                effect_size = (2 * stat_val) / (n1 * n2) - 1
                effect_size_type = "rank-biserial r"

        # ---- Multi-group case (>2 groups) ----
        elif n_groups > 2:
            if all_normal:
                levene_p = self.variance_homogeneity_test(group_data)
                if levene_p > self.alpha:
                    stat_val, p_value = stats.f_oneway(*group_data)
                    test_used = "One-way ANOVA"
                else:
                    # Design choice: use Kruskal-Wallis when variance is heterogeneous.
                    # Welch's ANOVA (alexandergovern) is theoretically better but
                    # Kruskal-Wallis is simpler and does not require extra packages.
                    stat_val, p_value = stats.kruskal(*group_data)
                    test_used = "Kruskal-Wallis 檢定 (方差不齊)"
            else:
                levene_p = 'N/A (常態性未通過)'
                stat_val, p_value = stats.kruskal(*group_data)
                test_used = "Kruskal-Wallis 檢定"

            # Effect size calculation
            if test_used == "One-way ANOVA":
                # Eta-squared: SS_between / SS_total
                all_vals = np.concatenate([d.values for d in group_data])
                grand_mean = all_vals.mean()
                ss_between = sum(len(d) * (d.mean() - grand_mean) ** 2 for d in group_data)
                ss_total = np.sum((all_vals - grand_mean) ** 2)
                effect_size = ss_between / ss_total if ss_total > 0 else 0.0
                effect_size_type = "η²"
            else:
                # Epsilon-squared for Kruskal-Wallis
                n_total = sum(len(d) for d in group_data)
                effect_size = (stat_val - n_groups + 1) / (n_total - n_groups)
                effect_size_type = "ε²"
        else:
            raise ValueError(f"至少需要 2 組資料，目前僅有 {n_groups} 組")

        # Format Shapiro results string for display
        shapiro_str = ' | '.join([f"{g}: {p}" for g, p in shapiro_results.items()])

        self.results.append({
            'Variable': var,
            'n_groups': n_groups,
            'Shapiro p-values': shapiro_str,
            'All Normal': '是' if all_normal else '否',
            "Levene p-value": levene_p,
            'Test Used': test_used,
            'Statistic': round(stat_val, 4),
            'p-value': p_value,
            'Effect Size': round(effect_size, 4),
            'Effect Size Type': effect_size_type,
            'Significant': '是' if p_value < self.alpha else '否',
        })

    def run_analysis(self, df, group_col, variables):
        """Run analysis for multiple variables with Bonferroni correction.

        Parameters
        ----------
        df : pd.DataFrame
        group_col : str
            Column name for grouping (e.g. 'Pclass', 'Survived').
        variables : list[str]
            Numeric columns to test.

        Returns
        -------
        pd.DataFrame
        """
        self.results = []  # Reset for fresh run

        for var in variables:
            self.analyze_variable(df, group_col, var)

        df_results = pd.DataFrame(self.results)

        if len(df_results) == 0:
            return df_results

        # Bonferroni correction
        pvals = df_results['p-value'].values
        _, pvals_corrected, _, _ = multipletests(pvals, alpha=self.alpha, method='bonferroni')
        df_results['p-value (Bonferroni)'] = pvals_corrected
        df_results['Significant (corrected)'] = [
            '是' if p < self.alpha else '否' for p in pvals_corrected
        ]

        return df_results

## 載入資料與生成 Codebook

使用 Titanic 資料集作為示範。`create_codebook()` 會自動識別數值型與類別型變數，並標記離群值。

In [ ]:
# Load data with relative path
df = pd.read_csv('./train.csv', encoding='utf-8-sig')

# Generate codebook
codebook = create_codebook(
    df,
    units_dict={'Age': 'years', 'Fare': 'USD'},
    detailed_labels={'Age': 'Passenger Age', 'Fare': 'Ticket Fare'},
)
codebook

## 範例 1：手動指定變數（2 組比較）

比較 **Survived = 0**（未生還）vs **Survived = 1**（生還）在 Fare 和 Age 上的差異。

由於只有 2 組，決策樹會判斷：
- Shapiro-Wilk → 常態性
- 若通過 → Levene → t 檢定或 Welch's t
- 若未通過 → Mann-Whitney U

In [ ]:
# 2-group comparison: Survived (0 vs 1)
module_2g = UnivariateAnalysisModule()
results_2g = module_2g.run_analysis(
    df=df,
    group_col='Survived',
    variables=['Fare', 'Age'],
)
results_2g

## 範例 2：多組比較（3 個艙等）

比較 **Pclass = 1, 2, 3** 在多個數值特徵上的差異。

使用 codebook 自動篩選 Scale 類型的欄位（排除 PassengerId, Survived, Pclass 本身）。

由於有 3 組，決策樹會判斷：
- Shapiro-Wilk **逐組檢定** → 全部通過才算常態
- 若常態 → Levene → ANOVA 或 Kruskal-Wallis
- 若非常態 → Kruskal-Wallis

In [ ]:
# Multi-group comparison: Pclass (1, 2, 3)
# Use codebook to find Scale variables, exclude ID/group columns
exclude_cols = {'PassengerId', 'Survived', 'Pclass'}
scale_vars = (
    codebook[codebook['Measure'] == 'Scale']['Name']
    .loc[lambda s: ~s.isin(exclude_cols)]
    .tolist()
)
print(f"分析變數: {scale_vars}")

module_mg = UnivariateAnalysisModule()
results_mg = module_mg.run_analysis(
    df=df,
    group_col='Pclass',
    variables=scale_vars,
)
results_mg

## 結果視覺化

### 基礎版：Boxplot + 檢定標註

以 Boxplot 呈現各組分佈，並在圖上標註使用的檢定方法、p 值與效應量。

### 進階版：多組比較 Pairwise 視覺化

當檢定為多組比較（ANOVA / Kruskal-Wallis）且結果顯著時，自動執行**事後配對比較**，
在圖上用連接線標註每對組別之間的 p 值，直觀呈現「到底哪兩組之間有差異」。

- 參數檢定 → Tukey HSD (`statsmodels`)
- 非參數檢定 → 逐對 Mann-Whitney U + Bonferroni 校正

In [ ]:
def _format_p(p):
    """Format p-value with significance stars."""
    if p < 0.001:
        return "p<.001 ***"
    elif p < 0.01:
        return f"p={p:.3f} **"
    elif p < 0.05:
        return f"p={p:.3f} *"
    else:
        return f"p={p:.3f} ns"


def _pairwise_tests(df, group_col, var, groups, is_parametric, alpha=0.05):
    """Run pairwise comparisons between all group pairs.

    Returns list of dicts: {pair, p_raw, p_corrected, significant, label}.
    """
    pairs = list(combinations(groups, 2))
    p_raw_list = []

    for g1, g2 in pairs:
        d1 = df[df[group_col] == g1][var].dropna()
        d2 = df[df[group_col] == g2][var].dropna()
        if is_parametric:
            _, p = stats.ttest_ind(d1, d2, equal_var=False)
        else:
            _, p = stats.mannwhitneyu(d1, d2, alternative='two-sided')
        p_raw_list.append(p)

    # Bonferroni correction across pairs
    n_pairs = len(pairs)
    p_corrected = [min(p * n_pairs, 1.0) for p in p_raw_list]

    results = []
    for (g1, g2), p_r, p_c in zip(pairs, p_raw_list, p_corrected):
        results.append({
            'pair': (g1, g2),
            'p_raw': p_r,
            'p_corrected': p_c,
            'significant': p_c < alpha,
            'label': _format_p(p_c),
        })
    return results


def _draw_bracket(ax, x1, x2, y, h, label, color='black'):
    """Draw a comparison bracket with p-value label between two x positions."""
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y],
            lw=1.2, color=color)
    ax.text((x1 + x2) / 2, y + h, label,
            ha='center', va='bottom', fontsize=8, color=color)


def plot_test_results(df, results_df, group_col, show_pairwise=True):
    """Create boxplots with hypothesis test results and optional pairwise brackets.

    Parameters
    ----------
    df : pd.DataFrame
    results_df : pd.DataFrame  — output of UnivariateAnalysisModule.run_analysis
    group_col : str
    show_pairwise : bool — if True, add pairwise comparison brackets for multi-group tests
    """
    n_vars = len(results_df)
    if n_vars == 0:
        print("No results to plot.")
        return

    fig, axes = plt.subplots(1, n_vars, figsize=(6 * n_vars, 6))
    if n_vars == 1:
        axes = [axes]

    groups = sorted(df[group_col].dropna().unique())
    group_pos = {g: i for i, g in enumerate(groups)}  # x-position mapping

    for ax, (_, row) in zip(axes, results_df.iterrows()):
        var = row['Variable']
        palette = sns.color_palette("Set2", n_colors=len(groups))
        sns.boxplot(data=df, x=group_col, y=var, ax=ax, palette=palette,
                    order=groups, fliersize=3)

        # Overlay strip plot for individual points
        sns.stripplot(data=df, x=group_col, y=var, ax=ax, order=groups,
                      color='black', alpha=0.25, size=2, jitter=True)

        # Build annotation title
        title_lines = [
            f"{var}",
            f"{row['Test Used']}",
            f"p = {row['p-value']:.4f}",
        ]
        es = row['Effect Size']
        if pd.notna(es) and es != 'N/A':
            title_lines.append(f"{row['Effect Size Type']} = {es:.3f}")

        sig = row['Significant'] == '是'
        color = 'red' if sig else 'black'
        ax.set_title('\n'.join(title_lines), fontsize=10, color=color,
                     fontweight='bold' if sig else 'normal')

        # ---- Pairwise brackets for multi-group comparisons ----
        n_grp = row['n_groups']
        if show_pairwise and n_grp > 2 and sig:
            is_param = 'ANOVA' in row['Test Used']
            pw = _pairwise_tests(df, group_col, var, groups, is_param)

            # Calculate bracket positions
            y_max = df[var].dropna().max()
            y_range = df[var].dropna().max() - df[var].dropna().min()
            bracket_gap = y_range * 0.06
            bracket_h = y_range * 0.02

            for k, comp in enumerate(pw):
                g1, g2 = comp['pair']
                x1 = group_pos[g1]
                x2 = group_pos[g2]
                y_pos = y_max + bracket_gap * (k + 1)
                bcolor = 'red' if comp['significant'] else 'gray'
                _draw_bracket(ax, x1, x2, y_pos, bracket_h,
                              comp['label'], color=bcolor)

            # Expand y-axis to fit brackets
            ax.set_ylim(top=y_max + bracket_gap * (len(pw) + 1.5))

        ax.set_xlabel(group_col, fontsize=11)
        ax.set_ylabel(var, fontsize=11)

    plt.tight_layout()
    plt.show()


# ---- Plot 2-group results ----
print("=== 2 組比較: Survived ===")
plot_test_results(df, results_2g, 'Survived')

# ---- Plot multi-group results with pairwise brackets ----
print("\n=== 多組比較: Pclass（含事後配對檢定）===")
plot_test_results(df, results_mg, 'Pclass', show_pairwise=True)

### 事後配對比較結果表

以下表格顯示 Pclass 多組比較中，每對組別的配對檢定結果（Bonferroni 校正後）。

In [ ]:
# 對每個顯著的多組變數，輸出 pairwise 比較結果
groups_pclass = sorted(df['Pclass'].dropna().unique())
pairwise_rows = []

for _, row in results_mg.iterrows():
    if row['n_groups'] > 2 and row['Significant'] == '是':
        var = row['Variable']
        is_param = 'ANOVA' in row['Test Used']
        pw = _pairwise_tests(df, 'Pclass', var, groups_pclass, is_param)
        for comp in pw:
            g1, g2 = comp['pair']
            pairwise_rows.append({
                'Variable': var,
                'Group 1': g1,
                'Group 2': g2,
                'p (raw)': round(comp['p_raw'], 6),
                'p (Bonferroni)': round(comp['p_corrected'], 6),
                'Significant': 'Yes ***' if comp['p_corrected'] < 0.001
                               else 'Yes **' if comp['p_corrected'] < 0.01
                               else 'Yes *' if comp['p_corrected'] < 0.05
                               else 'No',
            })

df_pairwise = pd.DataFrame(pairwise_rows)
df_pairwise

## 工具總結與注意事項

### 自動化的價值

- **一致性**：相同資料永遠產生相同結果，不受分析者主觀判斷影響
- **可重複性**：決策邏輯透明，任何人都能理解為何選擇特定檢定
- **效率**：一次分析多個變數，自動套用 Bonferroni 校正

### 限制與注意事項

1. **Shapiro-Wilk 的大樣本問題**：當 n > 5000 時，Shapiro-Wilk 幾乎必定拒絕常態性假設。此時可考慮改用 Q-Q plot 視覺判斷或 Kolmogorov-Smirnov 檢定。

2. **IQR 離群值與常態性是獨立判斷**：codebook 中的 OutlierWarning 基於 IQR 法則，與 Shapiro-Wilk 檢定結果無直接關聯。兩者應分別解讀。

3. **方差不齊時的多組比較**：本模組在常態但方差不齊時使用 Kruskal-Wallis，這是較保守的選擇。若需更精確的結果，可使用 Welch's ANOVA (`scipy.stats.alexandergovern` 或 `pingouin.welch_anova`)。

### 延伸方向

- **事後比較 (Post-hoc)**：當 ANOVA 或 Kruskal-Wallis 顯著時，進一步比較哪兩組之間有差異
  - 參數：Tukey HSD (`statsmodels.stats.multicomp.pairwise_tukeyhsd`)
  - 非參數：Dunn's test (`scikit-posthocs.posthoc_dunn`)
- **多變量分析**：MANOVA（多個依變數同時檢定）
- **效應量解讀標準化**：結合 bootstrapping 建立效應量的信賴區間